In [47]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore 
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel

In [49]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [50]:
loader=PyPDFLoader(
    "../data/medical_report.pdf",
)
docs=loader.load()


In [51]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
docs=splitter.split_documents(docs)

embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vector_store=InMemoryVectorStore.from_documents(
    documents=docs,
    embedding=embeddings
)



In [52]:
from pydantic import Field
class RagState(BaseModel):
    question:str=Field(description="User Question")
    documents:list=Field(default_factory=list)
    context: str=Field(description="Context data for user question",default="")
    answer:str=Field(description="Final answer",default="")
    

In [53]:
# question->retrive ->context->generate-> the end

def retrive_node(state:RagState)->RagState:
    docs=vector_store.similarity_search(query=state.question)
    state.documents=docs
    return state

def create_context_node(state:RagState)->RagState:
    context=""
    for doc in state.documents:
        context+=doc.page_content +"\n\n"
    state.context=context
    return state

def generate_node(state:RagState)->RagState:
    prompt=f"""
        you are an assistant and provide the answer for user question based on the 
        provided context if u dont find the releavant answer just say i dont know 
        context= {state.context},
        question={state.question}
    """
    res=llm.invoke(prompt)
    state.answer=res.content
    return state

In [54]:
graph=StateGraph(RagState)
graph.add_node("retrive_node",retrive_node)
graph.add_node("create_context_node",create_context_node)
graph.add_node("generate_node",generate_node)
graph.add_edge(START,"retrive_node")
graph.add_edge("retrive_node","create_context_node")
graph.add_edge("create_context_node","generate_node")
graph.add_edge("generate_node",END)

graph=graph.compile()

In [55]:
ans=graph.invoke({"question":"what is the name of patient"})


In [58]:
print(ans["answer"])

The patient’s name is **Ms. Nikita Chudhary**.
